# Buổi 4 — Xử lý file, logging và ảnh

**Thời lượng:** ~2 giờ (chiều ngày 2)

| Phần | Nội dung | Thời gian |
|---|---|---|
| A | Đường dẫn & duyệt thư mục ảnh (`pathlib`) | 15 phút |
| B | Đọc / ghi CSV — dữ liệu đo và báo cáo | 25 phút |
| C | Logging — nhật ký chạy máy | 30 phút |
| D | NumPy + ảnh — đọc, cắt ROI, ngưỡng, đo khuyết tật | 30 phút |
| E | Bài tập tổng hợp: trạm kiểm tra mini | 20 phút |

Cuối buổi bạn có một chương trình **chạy được thật**: quét thư mục ảnh → kiểm tra → ghi CSV + log + lưu ảnh NG.

In [ ]:
# === Ô CHUẨN BỊ: chạy ô này đầu tiên ===
# Tạo thư mục data/ (ảnh sản phẩm + file đo) nếu chưa có. Chạy được cả trên máy và trên Colab.
from pathlib import Path
import csv, random
import numpy as np
from PIL import Image

DATA = Path("data"); IMG_DIR = DATA / "images"; CSV_PATH = DATA / "measurements.csv"
NOMINAL, TOLERANCE = 50.0, 0.25

def _tao_du_lieu():
    rng, nprng = random.Random(26), np.random.default_rng(26)
    IMG_DIR.mkdir(parents=True, exist_ok=True)
    H, W = 240, 320
    yy, xx = np.mgrid[0:H, 0:W]
    rows = []
    for i in range(1, 13):
        pid = f"SP{i:03d}"
        img = nprng.normal(28, 4, size=(H, W))
        cy, cx = H // 2 + rng.randint(-6, 6), W // 2 + rng.randint(-8, 8)
        r0 = 85 + rng.randint(-3, 3)
        disk = (yy - cy) ** 2 + (xx - cx) ** 2 <= r0 ** 2
        img[disk] = nprng.normal(185, 5, size=disk.sum())
        for _ in range(2 if i in (3, 7) else (1 if i == 10 else 0)):
            a, d = rng.uniform(0, 6.28), rng.uniform(0, r0 * 0.75)
            dy, dx, r = int(cy + d * np.sin(a)), int(cx + d * np.cos(a)), rng.randint(5, 11)
            spot = ((yy - dy) ** 2 + (xx - dx) ** 2 <= r ** 2) & disk
            img[spot] = nprng.normal(55, 6, size=spot.sum())
        Image.fromarray(np.clip(img, 0, 255).astype(np.uint8)).save(IMG_DIR / f"{pid}.png")
        m = round(NOMINAL + nprng.normal(0, 0.08), 2)
        if i in (4, 9):
            m = round(NOMINAL + rng.choice([-1, 1]) * rng.uniform(0.3, 0.5), 2)
        rows.append({"product_id": pid, "line": rng.choice(["L1", "L2"]), "measured_mm": m,
                     "nominal_mm": NOMINAL, "tolerance_mm": TOLERANCE,
                     "operator": rng.choice(["A01", "A02", "B07"])})
    with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

if not CSV_PATH.exists():
    _tao_du_lieu()
print("Dữ liệu sẵn sàng:", len(list(IMG_DIR.glob("*.png"))), "ảnh,", CSV_PATH)


## A. Đường dẫn và duyệt thư mục

Dùng `pathlib` thay cho nối chuỗi `"data" + "/" + "images"` — chạy đúng trên cả Windows và Linux (máy trạm trong xưởng thường là Windows, server lại là Linux).

In [ ]:
from pathlib import Path

thu_muc_anh = Path("data") / "images"

print("Thư mục có tồn tại?", thu_muc_anh.exists())
print("Đường dẫn tuyệt đối:", thu_muc_anh.resolve())

danh_sach_anh = sorted(thu_muc_anh.glob("*.png"))      # chỉ lấy file .png, sắp xếp theo tên
print("Số ảnh:", len(danh_sach_anh))

p = danh_sach_anh[0]
print("Tên file     :", p.name)
print("Tên không đuôi:", p.stem)        # SP001 -> chính là mã sản phẩm
print("Phần mở rộng :", p.suffix)

In [ ]:
# Tạo thư mục kết quả (không lỗi nếu đã có)
KET_QUA = Path("ket_qua")
(KET_QUA / "anh_ng").mkdir(parents=True, exist_ok=True)
print("Đã chuẩn bị:", KET_QUA.resolve())

# Đặt tên file theo ngày - rất quan trọng khi chạy nhiều ca
from datetime import datetime
hom_nay = datetime.now().strftime("%Y%m%d")
file_bao_cao = KET_QUA / f"ket_qua_{hom_nay}.csv"
print("File báo cáo hôm nay:", file_bao_cao.name)

## B. Đọc và ghi CSV

CSV là ngôn ngữ chung giữa Python, Excel và hệ thống MES.

In [ ]:
import csv

with open(CSV_PATH, encoding="utf-8") as f:       # with ... -> tự đóng file, kể cả khi lỗi
    bang_do = list(csv.DictReader(f))              # mỗi dòng thành một dict

print("Số dòng:", len(bang_do))
print("Các cột:", list(bang_do[0].keys()))
print("Dòng đầu:", bang_do[0])

> **Bẫy số 1 khi đọc CSV:** mọi giá trị đọc lên đều là **chuỗi**.
> `bang_do[0]["measured_mm"]` là `"50.1"` chứ không phải `50.1` → phải `float(...)` trước khi tính.

> **Bẫy số 2 — tiếng Việt:** luôn ghi rõ `encoding="utf-8"`. Nếu file do Excel tiếng Việt xuất ra mà bị lỗi font, thử `encoding="utf-8-sig"` hoặc `encoding="cp1258"`.

In [ ]:
# Tính toán rồi ghi kết quả ra CSV mới
ket_qua = []
for dong in bang_do:
    do_duoc = float(dong["measured_mm"])
    sai_lech = do_duoc - float(dong["nominal_mm"])
    dat = abs(sai_lech) <= float(dong["tolerance_mm"])
    ket_qua.append({
        "product_id": dong["product_id"],
        "line": dong["line"],
        "measured_mm": do_duoc,
        "deviation_mm": round(sai_lech, 3),
        "result": "OK" if dat else "NG",
    })

with open(file_bao_cao, "w", newline="", encoding="utf-8") as f:   # newline="" để Excel không chèn dòng trống
    writer = csv.DictWriter(f, fieldnames=list(ket_qua[0].keys()))
    writer.writeheader()
    writer.writerows(ket_qua)

print("Đã ghi", len(ket_qua), "dòng vào", file_bao_cao)
print(file_bao_cao.read_text(encoding="utf-8")[:220], "...")

> **Khi nào dùng `pandas`?** Khi phải lọc, nhóm, thống kê nhiều: `pd.read_csv(...)` rồi `df.groupby("line")["result"].value_counts()`.
> Buổi này cố tình dùng `csv` thuần để bạn thấy rõ dữ liệu đi qua từng bước — xử lý ảnh realtime cũng ít khi cần pandas.

## C. Logging — nhật ký chạy máy

`print()` để học. Phần mềm chạy ở xưởng phải dùng `logging`, vì:

| | `print()` | `logging` |
|---|---|---|
| Thời gian | không có | có timestamp từng dòng |
| Mức độ | tất cả như nhau | DEBUG / INFO / WARNING / ERROR / CRITICAL |
| Lưu lại | mất khi tắt máy | ghi ra file, xoay vòng theo dung lượng |
| Tắt bớt | phải xóa code | đổi 1 dòng cấu hình |

Khi sản phẩm bị khách trả về sau 3 tháng, **file log là bằng chứng duy nhất** cho biết hôm đó trạm đã kết luận gì.

In [ ]:
import logging
from logging.handlers import RotatingFileHandler

def tao_logger(ten="tram_kiem_tra", file_log=KET_QUA / "tram.log"):
    """Tạo logger ghi đồng thời ra màn hình và ra file xoay vòng."""
    logger = logging.getLogger(ten)
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()        # tránh ghi trùng khi chạy lại ô này trong notebook

    dinh_dang = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    ra_man_hinh = logging.StreamHandler()
    ra_man_hinh.setLevel(logging.INFO)            # màn hình: gọn, chỉ từ INFO trở lên
    ra_man_hinh.setFormatter(dinh_dang)

    ra_file = RotatingFileHandler(file_log, maxBytes=1_000_000, backupCount=5, encoding="utf-8")
    ra_file.setLevel(logging.DEBUG)               # file: ghi đầy đủ để truy vết
    ra_file.setFormatter(dinh_dang)

    logger.addHandler(ra_man_hinh)
    logger.addHandler(ra_file)
    return logger

log = tao_logger()
log.info("Trạm kiểm tra khởi động")

In [ ]:
# 5 mức và cách dùng trong nhà máy
log.debug("Ảnh SP001: shape=(240, 320), thời gian xử lý 12 ms")   # chi tiết kỹ thuật, chỉ vào file
log.info("SP001: OK, sai lệch +0.10 mm")                          # diễn biến bình thường
log.warning("SP007: sai lệch +0.23 mm, sát ngưỡng dung sai")      # chưa lỗi nhưng cần để mắt
log.error("Không đọc được ảnh SP013.png - bỏ qua sản phẩm")       # một việc thất bại
log.critical("Mất kết nối camera - dừng dây chuyền")              # phải can thiệp ngay

In [ ]:
# Ghi lại lỗi kèm nguyên nhân: logger.exception() lưu cả traceback vào file
def doc_gia_tri(chuoi):
    return float(chuoi)

for gia_tri in ["50.1", "50,1"]:          # dấu phẩy thay vì dấu chấm -> lỗi kinh điển khi nhập tay
    try:
        print("Đọc được:", doc_gia_tri(gia_tri))
    except ValueError:
        log.exception("Giá trị đo không hợp lệ: %r", gia_tri)

In [ ]:
# Xem lại file log vừa ghi
print((KET_QUA / "tram.log").read_text(encoding="utf-8")[-900:])

**Quy ước log nên áp dụng ngay:**

- Mỗi sản phẩm **một dòng INFO**: mã, kết luận, số đo chính, thời gian chu kỳ.
- Mỗi lần đổi tham số (dung sai, ngưỡng sáng) → một dòng WARNING kèm người đổi.
- Không log mật khẩu, không log đường dẫn chứa thông tin khách hàng.
- `maxBytes` + `backupCount`: máy trạm chạy 24/7, không giới hạn là **đầy ổ cứng sau vài tháng**.

## D. NumPy và ảnh

Ảnh = mảng NumPy. Mọi thao tác kiểm tra đều là phép tính trên mảng đó.

In [ ]:
import numpy as np

a = np.array([[12, 15, 18, 20],
              [11, 14, 17, 21],
              [10, 13, 16, 19]])

print("shape:", a.shape, "| dtype:", a.dtype)
print("phần tử hàng 1 cột 2:", a[1, 2])
print("cả cột 1            :", a[:, 1])
print("vùng 2x2 góc trên   :\n", a[0:2, 0:2])
print("cộng 10 cho tất cả  :\n", a + 10)
print("trung bình / lớn nhất:", a.mean(), a.max())
print("mặt nạ a > 15       :\n", a > 15)
print("đếm phần tử > 15    :", (a > 15).sum())

In [ ]:
# Đọc ảnh. Dùng OpenCV nếu có, không thì dùng Pillow - kết quả đều là mảng NumPy.
try:
    import cv2
except ImportError:
    cv2 = None
    print("Chưa có OpenCV -> dùng Pillow. Cài bằng: pip install opencv-python")

from PIL import Image

def doc_anh_xam(duong_dan):
    """Đọc ảnh xám, trả về mảng NumPy uint8. Trả None nếu không đọc được."""
    duong_dan = str(duong_dan)
    if cv2 is not None:
        return cv2.imread(duong_dan, cv2.IMREAD_GRAYSCALE)
    anh = Image.open(duong_dan).convert("L")
    return np.array(anh)

anh = doc_anh_xam(IMG_DIR / "SP003.png")

if anh is None:                       # cv2.imread KHÔNG báo lỗi khi sai đường dẫn, chỉ trả None
    log.error("Không đọc được ảnh")
else:
    print("shape:", anh.shape, "| dtype:", anh.dtype, "| min/max:", anh.min(), anh.max())

> **Ba điều phải nhớ với OpenCV:**
> 1. `cv2.imread` trả `None` khi sai đường dẫn — **luôn kiểm tra `if anh is None`**.
> 2. Ảnh màu đọc theo thứ tự **BGR**, không phải RGB. Muốn hiển thị bằng matplotlib: `cv2.cvtColor(anh, cv2.COLOR_BGR2RGB)`.
> 3. Chỉ số là `anh[y, x]` — **hàng trước, cột sau**, ngược với thói quen (x, y) trên bản vẽ.

In [ ]:
import matplotlib.pyplot as plt

NGUONG_SANG = 110
MM_PER_PIXEL = 0.1
GIOI_HAN_LOI = 3.0

from scipy import ndimage

vung_sang = anh > NGUONG_SANG                      # mặt nạ: True ở pixel sáng
san_pham = ndimage.binary_fill_holes(vung_sang)    # lấp lỗ -> toàn bộ thân sản phẩm
loi = san_pham & ~vung_sang                        # vết tối nằm trong thân = khuyết tật

dien_tich_mm2 = loi.sum() * MM_PER_PIXEL ** 2
ket_luan = "NG" if dien_tich_mm2 > GIOI_HAN_LOI else "OK"
print(f"Số pixel lỗi: {loi.sum()} -> {dien_tich_mm2:.2f} mm2 -> {ket_luan}")

fig, ax = plt.subplots(1, 3, figsize=(9, 2.6))
ax[0].imshow(anh, cmap="gray", vmin=0, vmax=255); ax[0].set_title("ảnh gốc")
ax[1].imshow(san_pham, cmap="gray");              ax[1].set_title("vùng sản phẩm")
ax[2].imshow(loi, cmap="hot");                    ax[2].set_title(f"khuyết tật {dien_tich_mm2:.2f} mm2")
for t in ax: t.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Cắt ROI: chỉ kiểm tra vùng quan tâm -> nhanh hơn và ít báo giả hơn
roi = anh[60:180, 80:240]        # [hàng bắt đầu:hàng kết thúc, cột bắt đầu:cột kết thúc]
print("ROI shape:", roi.shape, "| độ sáng trung bình:", round(roi.mean(), 1))

# Khoanh vùng khuyết tật bằng np.where: lấy tọa độ các pixel lỗi
ys, xs = np.where(loi)
if len(ys) > 0:
    y1, y2, x1, x2 = ys.min(), ys.max(), xs.min(), xs.max()
    print(f"Hộp bao khuyết tật: x {x1}-{x2}, y {y1}-{y2}")

    anh_danh_dau = np.dstack([anh, anh, anh])          # xám -> 3 kênh để vẽ màu lên
    anh_danh_dau[y1:y2+1, [x1, x2]] = [255, 0, 0]      # 2 cạnh dọc
    anh_danh_dau[[y1, y2], x1:x2+1] = [255, 0, 0]      # 2 cạnh ngang

    plt.figure(figsize=(4, 3)); plt.imshow(anh_danh_dau); plt.axis("off")
    plt.title("Vị trí khuyết tật"); plt.show()

In [ ]:
# Lưu ảnh NG để truy vết sau này
duong_dan_luu = KET_QUA / "anh_ng" / "SP003_ng.png"
Image.fromarray(anh_danh_dau.astype(np.uint8)).save(duong_dan_luu)   # hoặc cv2.imwrite(str(duong_dan_luu), anh)
log.info("Đã lưu ảnh NG: %s", duong_dan_luu)

## E. Bài tập tổng hợp — Trạm kiểm tra mini (20 phút)

Viết một vòng lặp quét **toàn bộ** thư mục `data/images`, với mỗi ảnh:

1. Đọc ảnh, nếu đọc lỗi thì `log.error` và bỏ qua.
2. Tính diện tích khuyết tật, kết luận OK/NG.
3. Lấy thêm số đo kích thước của mã sản phẩm đó từ `measurements.csv` và kiểm tra dung sai.
4. Kết luận cuối: NG nếu **một trong hai** tiêu chí không đạt; ghi `log.info` mỗi sản phẩm.
5. Ghi toàn bộ kết quả ra `ket_qua/bao_cao_cuoi.csv`.
6. Ảnh nào NG thì lưu vào `ket_qua/anh_ng/`.
7. In tổng kết: số OK, số NG, tỉ lệ NG, nguyên nhân NG hay gặp nhất.

In [ ]:
# TODO: làm ở đây


<details><summary>Gợi ý lời giải</summary>

```python
# Tra cứu số đo theo mã sản phẩm
so_do = {d["product_id"]: d for d in bang_do}

bao_cao, dem_nguyen_nhan = [], {}

for duong_dan in sorted(IMG_DIR.glob("*.png")):
    ma = duong_dan.stem
    anh = doc_anh_xam(duong_dan)
    if anh is None:
        log.error("Không đọc được ảnh %s", duong_dan.name)
        continue

    vung_sang = anh > NGUONG_SANG
    loi = ndimage.binary_fill_holes(vung_sang) & ~vung_sang
    dien_tich = loi.sum() * MM_PER_PIXEL ** 2

    dong_do = so_do.get(ma)
    sai_lech = float(dong_do["measured_mm"]) - float(dong_do["nominal_mm"]) if dong_do else 0.0
    dung_sai = float(dong_do["tolerance_mm"]) if dong_do else 0.25

    ly_do = []
    if dien_tich > GIOI_HAN_LOI:
        ly_do.append("be_mat")
    if abs(sai_lech) > dung_sai:
        ly_do.append("kich_thuoc")
    ket_luan = "NG" if ly_do else "OK"

    log.info("%s: %s | sai lệch %+.2f mm | lỗi %.2f mm2 %s",
             ma, ket_luan, sai_lech, dien_tich, ("| " + ", ".join(ly_do)) if ly_do else "")

    for nn in ly_do:
        dem_nguyen_nhan[nn] = dem_nguyen_nhan.get(nn, 0) + 1
    if ket_luan == "NG":
        Image.fromarray(anh).save(KET_QUA / "anh_ng" / f"{ma}_ng.png")

    bao_cao.append({"product_id": ma, "deviation_mm": round(sai_lech, 3),
                    "defect_mm2": round(dien_tich, 2), "result": ket_luan,
                    "reason": ",".join(ly_do)})

with open(KET_QUA / "bao_cao_cuoi.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(bao_cao[0].keys()))
    w.writeheader(); w.writerows(bao_cao)

so_ng = sum(1 for d in bao_cao if d["result"] == "NG")
print(f"\nTổng {len(bao_cao)} sản phẩm | OK {len(bao_cao) - so_ng} | NG {so_ng} "
      f"| tỉ lệ NG {so_ng / len(bao_cao):.1%}")
print("Nguyên nhân:", sorted(dem_nguyen_nhan.items(), key=lambda x: -x[1]))
```
</details>

---

## Chốt khóa học

Bạn đã đi hết một vòng của trạm kiểm tra thật:

```
ảnh/dữ liệu vào  ->  tính toán (NumPy)  ->  so ngưỡng (if)  ->  kết luận OK/NG
                                                              ->  ghi CSV cho MES
                                                              ->  ghi log để truy vết
                                                              ->  lưu ảnh NG làm bằng chứng
```

**Việc nên làm ngay tuần sau:** lấy một bài kiểm tra có thật ở xưởng, chụp 20 ảnh mẫu (10 OK, 10 NG), và viết lại đúng vòng lặp ở phần E với số liệu của mình.

**Hướng đi tiếp:**

- `pandas` + biểu đồ: báo cáo chất lượng theo ca, theo chuyền.
- OpenCV nâng cao: căn chỉnh vị trí, template matching, đo kích thước bằng ảnh.
- PyQt6/PySide6: dựng giao diện HMI cho người vận hành (xem deck PyQt6).
- Khi luật không đủ: học máy / deep learning cho khuyết tật khó mô tả.